# Colab A100 — KOSIS 재임베딩 워커 (COLAB_A100, 55,000건)

**VS Code에서 Colab 런타임에 붙여 실행하는 버전** — Colab Secrets(`userdata`)는 Colab 웹 UI에서만 동작하므로 쓰지 않고, 실행 시 직접 입력받습니다(화면에 안 보임).

실행 전 7-1 서버 터미널에서 아래 3개를 미리 복사해두세요:
```bash
# 1) SSH 개인키 (base64 한 줄)
base64 -w0 /home/ubuntu/.ssh/colab_tunnel/colab_a100_key; echo

# 2) DB 비밀번호 / 3) KOSIS_API_KEY_2 -- .env 파일에서 확인
grep -E '^KOSIS_API_KEY_2|^SUPABASE_DB_URL' /home/ubuntu/ai-nlp-project-reembedding/.env
```

담당 범위: `line_no 88749~143748` (COLAB_A100 파티션, SERVER_A/B와 겹치지 않음)

## 셀 1: 환경설정 + 자격증명 입력

In [ ]:
import os, base64
from getpass import getpass

# --- SSH 개인키 (base64 한 줄로 붙여넣기) ---
key_b64 = getpass("SSH 개인키 base64 한 줄 붙여넣기: ").strip()
os.makedirs("/content/.ssh", exist_ok=True)
with open("/content/.ssh/colab_a100_key", "wb") as f:
    f.write(base64.b64decode(key_b64))
os.chmod("/content/.ssh/colab_a100_key", 0o600)

# 키가 제대로 복원됐는지 형식만 확인(내용은 출력 안 함)
with open("/content/.ssh/colab_a100_key") as f:
    head = f.readline().strip()
assert "BEGIN OPENSSH PRIVATE KEY" in head, f"키 형식 이상: {head[:40]}"
print("SSH 키 복원 완료")

# --- 나머지 자격증명 ---
os.environ["DB_USER"] = "kosis_user"
os.environ["DB_NAME"] = "kosis_db"
os.environ["DB_PASSWORD"] = getpass("DB 비밀번호: ")
os.environ["KOSIS_API_KEY_2"] = getpass("KOSIS_API_KEY_2: ")

# --- 접속/실행 설정 ---
os.environ["SSH_HOST"] = "51.20.253.79"
os.environ["COLAB_SSH_KEY_PATH"] = "/content/.ssh/colab_a100_key"
os.environ["LOCAL_DB_PORT"] = "15432"
os.environ["HF_HOME"] = "/content/hf_cache"

!pip install -q psycopg2-binary sentence-transformers
!mkdir -p agent/kosis agent/preprocessing benchmark
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("환경 준비 완료")

## 셀 2: SSH 터널 스크립트 생성

In [ ]:
%%writefile agent/kosis/colab_ssh_tunnel.sh
#!/usr/bin/env bash
# Colab -> 7-1 PostgreSQL(5432) SSH 터널. 이미 열려있는 SSH(22)를 재사용하며
# 새 포트를 열지 않는다. 사용 키는 authorized_keys에 restrict,port-forwarding으로
# 제한돼 셸 접속은 불가하고 포트포워딩만 가능(7-1에서 실측 검증 완료).
set -euo pipefail

SSH_HOST="${SSH_HOST:-51.20.253.79}"
SSH_USER="${SSH_USER:-ubuntu}"
LOCAL_DB_PORT="${LOCAL_DB_PORT:-15432}"
KEY_PATH="${COLAB_SSH_KEY_PATH:-/content/.ssh/colab_a100_key}"

if [ ! -f "$KEY_PATH" ]; then
    echo "[오류] 개인키 파일이 없습니다: $KEY_PATH" >&2
    exit 1
fi
chmod 600 "$KEY_PATH"

pkill -f "ssh .*-L ${LOCAL_DB_PORT}:127.0.0.1:5432" 2>/dev/null || true
sleep 1

ssh -f -N \
    -o StrictHostKeyChecking=no \
    -o ServerAliveInterval=30 \
    -o ServerAliveCountMax=3 \
    -o ExitOnForwardFailure=yes \
    -i "$KEY_PATH" \
    -L "${LOCAL_DB_PORT}:127.0.0.1:5432" \
    "${SSH_USER}@${SSH_HOST}"

sleep 2

python3 - "$LOCAL_DB_PORT" <<'PYEOF'
import socket, sys
port = int(sys.argv[1])
s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.settimeout(3)
try:
    s.connect(("127.0.0.1", port))
    print(f"[OK] 로컬 포트 {port} 터널 정상 리스닝 중")
except Exception as e:
    print(f"[오류] 로컬 포트 {port} 연결 실패: {e}")
    sys.exit(1)
finally:
    s.close()
PYEOF

echo "터널 준비 완료: 127.0.0.1:${LOCAL_DB_PORT} -> 7-1(${SSH_HOST}):5432"

## 셀 3: 터널 실행 + DB 연결 확인

In [ ]:
!bash agent/kosis/colab_ssh_tunnel.sh

import os, psycopg2

os.environ["SUPABASE_DB_URL"] = (
    f'postgresql://{os.environ["DB_USER"]}:{os.environ["DB_PASSWORD"]}'
    f'@127.0.0.1:{os.environ["LOCAL_DB_PORT"]}/{os.environ["DB_NAME"]}'
)
conn = psycopg2.connect(os.environ["SUPABASE_DB_URL"])
cur = conn.cursor()
cur.execute("select 1")
print("DB 터널 연결 성공:", cur.fetchone())

cur.execute("""
    select status, count(*), min(line_no), max(line_no)
    from kosis_reembed_checkpoint_qwen where server_role='COLAB_A100' group by status
""")
print("COLAB_A100 담당 범위:")
for row in cur.fetchall():
    print(" ", row)
conn.close()

## 셀 4: 필요한 파일 수신 (GitHub 미사용, DB 경유)

In [ ]:
import psycopg2, base64, hashlib, os

conn = psycopg2.connect(os.environ["SUPABASE_DB_URL"])
cur = conn.cursor()

files_needed = [
    "agent/kosis/crawl_output/tables.jsonl",
    "agent/preprocessing/kosis_org_whitelist.json",
    "agent/kosis/reembed_worker.py",
    "benchmark/metadata_embedding_experiment.py",
    "agent/kosis/reembed_worker_colab.py",
]

for fname in files_needed:
    cur.execute("select content_b64, md5 from tmp_file_transfer where filename=%s", (fname,))
    row = cur.fetchone()
    if row is None:
        raise SystemExit(f"7-1에서 {fname} 전달이 아직 안 됐습니다.")
    b64, md5 = row
    data = base64.b64decode(b64)
    assert hashlib.md5(data).hexdigest() == md5, f"{fname} 체크섬 불일치!"
    os.makedirs(os.path.dirname(fname) or ".", exist_ok=True)
    with open(fname, "wb") as f:
        f.write(data)
    print(f"받음: {fname} ({len(data)/1024/1024:.1f}MB)")

conn.close()

for p in ("agent/__init__.py", "agent/kosis/__init__.py",
          "agent/preprocessing/__init__.py", "benchmark/__init__.py"):
    open(p, "a").close()

with open("agent/kosis/crawl_output/tables.jsonl") as f:
    n = sum(1 for _ in f)
print("tables.jsonl 라인 수:", n, "(기대값: 287498)")
assert n == 287498

## 셀 5: 워커 실행 (COLAB_A100, 55,000건)

⚠️ 실행 전 7-1에서 SERVER_A가 `KOSIS_API_KEY_2`를 반납했는지(키 1개로 축소했는지) 확인하세요.
같은 키를 두 곳에서 동시에 쓰면 KOSIS rate limit에 걸립니다.

In [ ]:
import subprocess, os

env = os.environ.copy()
logf = open("colab_a100.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "agent.kosis.reembed_worker_colab", "COLAB_A100",
     "--limit", "55000", "--concurrency", "6",
     "--api-keys", os.environ["KOSIS_API_KEY_2"]],
    stdout=logf, stderr=subprocess.STDOUT, env=env,
)
print("백그라운드 실행 시작. PID:", proc.pid)

## 셀 6: 모니터링 (반복 실행 가능)

In [ ]:
!tail -30 colab_a100.log
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv

In [ ]:
import psycopg2, os
conn = psycopg2.connect(os.environ["SUPABASE_DB_URL"])
cur = conn.cursor()
cur.execute("""
    select server_role, status, count(*), min(line_no), max(line_no)
    from kosis_reembed_checkpoint_qwen
    where server_role in ('SERVER_A','SERVER_B','COLAB_A100')
    group by server_role, status order by server_role, status
""")
for row in cur.fetchall():
    print(row)

cur.execute("""
    select count(*) from (
        select table_id from kosis_reembed_checkpoint_qwen
        group by table_id having count(distinct server_role) > 1
    ) t
""")
print("파티션 겹침(0이어야 정상):", cur.fetchone()[0])
conn.close()